<!-- cspell:ignore colorbar dalitz fromkeys Kolmogorov linalg mystnb pvalue samp Smirnov -->

# Phase space generators

:::{autolink-concat}
:::

Amplitude analysis needs samples of four-momenta that are distributed uniformly over the phase space of a decay. {class}`.TFPhaseSpaceGenerator` produces such samples with [`phasespace`](https://phasespace.readthedocs.io). That package generates *weighted* events, so {class}`.TFPhaseSpaceGenerator` turns them into unweighted events with a hit-and-miss step: an event is kept if a uniform random number is below its weight. This page shows how the `adaptive_max_weight` option makes that step faster without changing the generated distribution.

In [ ]:
import logging
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats
from scipy.stats import ks_2samp

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("tensorwaves").setLevel(logging.ERROR)
set_matplotlib_formats("svg")

## Hit-and-miss against an estimated maximum weight

By default, the random numbers of the hit-and-miss step are drawn up to $1$, the upper bound of the normalized weights that `phasespace` guarantees. The weights that actually occur are often far below that bound, especially for decays into many particles, so most events are rejected. With `adaptive_max_weight=True`, the random numbers are drawn up to an estimate of the maximum weight instead. The estimate is the largest weight seen so far plus a margin of 5%. If a later bunch contains a larger weight, the estimate is raised and the sample is generated anew, so that the result remains unbiased.

The option is keyword-only and is off by default, so existing code keeps producing the same events for the same seed.

In [ ]:
from tensorwaves.data import TFPhaseSpaceGenerator, TFUniformRealNumberGenerator

initial_state_mass = 3.0969
final_state_masses = {0: 0.1350, 1: 0.1396, 2: 0.4937}
default_generator = TFPhaseSpaceGenerator(initial_state_mass, final_state_masses)
adaptive_generator = TFPhaseSpaceGenerator(
    initial_state_mass, final_state_masses, adaptive_max_weight=True
)

## Speed

The gain grows with the number of final state particles, because the spread of the weights grows with it. The following cell times both settings for a $5\,\mathrm{GeV}$ particle that decays into an increasing number of pions. These timings only give an impression on the machine that builds this page.

In [ ]:
def time_generation(n_pions: int, adaptive: bool, size: int = 10_000) -> float:
    generator = TFPhaseSpaceGenerator(
        initial_state_mass=5.0,
        final_state_masses=dict.fromkeys(range(n_pions), 0.13957),
        adaptive_max_weight=adaptive,
    )
    generator.show_progress = False
    rng = TFUniformRealNumberGenerator(seed=0)
    start = time.perf_counter()
    generator.generate(size, rng)
    return time.perf_counter() - start


time_generation(3, adaptive=False)  # warm up TensorFlow
timings = pd.DataFrame({
    "Default": {n: time_generation(n, adaptive=False) for n in range(3, 7)},
    "adaptive_max_weight=True": {
        n: time_generation(n, adaptive=True) for n in range(3, 7)
    },
})
timings.index.name = "Pions"
timings["Speed-up"] = timings["Default"] / timings["adaptive_max_weight=True"]
timings.style.format({
    "Default": "{:.2f} s",
    "adaptive_max_weight=True": "{:.2f} s",
    "Speed-up": "{:.1f}x",
})

For three particles, a single bunch of 50,000 events already contains enough accepted events, so both settings take about equally long. For more particles, the default setting needs many more bunches.

## Generated distributions

We now generate a large sample of the three-body decay with both settings. The events differ, because the hit-and-miss step consumes the random numbers differently, but both samples should be distributed uniformly over phase space. The Dalitz plots show this: the events fill the kinematically allowed region evenly with both settings.

In [ ]:
size = 500_000
default_generator.show_progress = False
adaptive_generator.show_progress = False
samples = {
    "Default": default_generator.generate(size, TFUniformRealNumberGenerator(seed=0)),
    "adaptive_max_weight=True": adaptive_generator.generate(
        size, TFUniformRealNumberGenerator(seed=0)
    ),
}

In [ ]:
def invariant_mass_squared(*momenta: np.ndarray) -> np.ndarray:
    p = np.sum(momenta, axis=0)
    return p[:, 0] ** 2 - np.sum(p[:, 1:] ** 2, axis=1)


variables = {
    label: pd.DataFrame({
        "m12": invariant_mass_squared(sample["p0"], sample["p1"]),
        "m23": invariant_mass_squared(sample["p1"], sample["p2"]),
        "m13": invariant_mass_squared(sample["p0"], sample["p2"]),
        "cos_theta": sample["p0"][:, 3] / np.linalg.norm(sample["p0"][:, 1:], axis=1),
    })
    for label, sample in samples.items()
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True, layout="constrained")
for ax, (label, df) in zip(axes, variables.items(), strict=True):
    ax.hist2d(df["m12"], df["m23"], bins=150, cmin=1, rasterized=True)
    ax.set_title(label)
    ax.set_xlabel(R"$m^2_{12}$ [GeV$^2$]")
axes[0].set_ylabel(R"$m^2_{23}$ [GeV$^2$]")
fig.colorbar(axes[0].collections[0], ax=axes, label="Events")
plt.show()

## Comparing the distributions

Overlaying the projections of both samples shows that they agree within statistical fluctuations. The lower panels show the difference per bin, divided by its statistical uncertainty. These pulls should scatter around zero with a spread of about one.

In [ ]:
labels = {
    "m12": R"$m^2_{12}$ [GeV$^2$]",
    "m23": R"$m^2_{23}$ [GeV$^2$]",
    "m13": R"$m^2_{13}$ [GeV$^2$]",
    "cos_theta": R"$\cos\theta_0$",
}
fig, axes = plt.subplots(
    2,
    len(labels),
    figsize=(13, 4.5),
    height_ratios=[3, 1],
    sharex="col",
    layout="constrained",
)
default, adaptive = variables.values()
for (top, bottom), (key, xlabel) in zip(axes.T, labels.items(), strict=True):
    bins = np.linspace(default[key].min(), default[key].max(), 61)
    n_default, _ = np.histogram(default[key], bins)
    n_adaptive, _ = np.histogram(adaptive[key], bins)
    for label, df in variables.items():
        top.hist(df[key], bins, histtype="step", label=label)
    pulls = (n_adaptive - n_default) / np.sqrt(n_adaptive + n_default)
    bottom.stairs(pulls, bins, fill=True, alpha=0.6)
    bottom.axhline(0, color="black", linewidth=0.5)
    bottom.set_ylim(-4, 4)
    bottom.set_xlabel(xlabel)
axes[0, 0].set_ylabel("Events")
axes[1, 0].set_ylabel("Pull")
axes[0, -1].legend(loc="lower center", fontsize="small")
plt.show()

A two-sample [Kolmogorov–Smirnov test](https://en.wikipedia.org/wiki/Kolmogorov%E2%80%93Smirnov_test) quantifies this. A $p$-value well above a typical threshold such as $0.05$ means that the test finds no evidence that the two samples come from different distributions. The table also lists the means and standard deviations, which agree to the precision expected for this sample size.

In [ ]:
pd.DataFrame({
    key: {
        "Mean (default)": default[key].mean(),
        "Mean (adaptive)": adaptive[key].mean(),
        "Std (default)": default[key].std(),
        "Std (adaptive)": adaptive[key].std(),
        "KS p-value": ks_2samp(default[key], adaptive[key]).pvalue,
    }
    for key in labels
}).T.style.format("{:.4f}")

The same holds for decays into more particles, where the option makes the biggest difference. The comparison below repeats the test for a five-pion decay, using the invariant mass of each pair of neighboring pions.

In [ ]:
n_pions = 5
five_pion_samples = {}
for adaptive in [False, True]:
    generator = TFPhaseSpaceGenerator(
        initial_state_mass=5.0,
        final_state_masses=dict.fromkeys(range(n_pions), 0.13957),
        adaptive_max_weight=adaptive,
    )
    generator.show_progress = False
    five_pion_samples[adaptive] = generator.generate(
        100_000, TFUniformRealNumberGenerator(seed=1)
    )

pairs = [(i, i + 1) for i in range(n_pions - 1)]
fig, axes = plt.subplots(1, len(pairs), figsize=(13, 3), layout="constrained")
p_values = {}
for ax, (i, j) in zip(axes, pairs, strict=True):
    m2 = {
        adaptive: invariant_mass_squared(sample[f"p{i}"], sample[f"p{j}"])
        for adaptive, sample in five_pion_samples.items()
    }
    bins = np.linspace(0, m2[False].max(), 50)
    ax.hist(m2[False], bins, histtype="step", label="Default")
    ax.hist(m2[True], bins, histtype="step", label="adaptive_max_weight=True")
    ax.set_xlabel(Rf"$m^2_{{{i}{j}}}$ [GeV$^2$]")
    p_values[f"m2_{i}{j}"] = ks_2samp(m2[False], m2[True]).pvalue
axes[0].set_ylabel("Events")
axes[-1].legend(fontsize="small")
plt.show()
pd.Series(p_values, name="KS p-value").to_frame().style.format("{:.3f}")